In [ ]:
import pandas as pd
import re

def ground_truths_escenario5(log_text):
    """
    Define la verdad para el Escenario 5: Accesos a /etc/shadow.
    Distingue ataques reales (exit=-13) de lecturas legítimas del sistema.
    """
    log = str(log_text).upper()
    
    # Extraer el AUID numérico para no depender de nombres en el código
    match_auid = re.search(r'AUID="?(\d+)"?', log)
    auid_num = int(match_auid.group(1)) if match_auid else -1
    
    # 1. ATAQUE REAL: Un usuario sin privilegios intenta leer/copiar el shadow y falla (exit=-13)
    if 'TYPE=SYSCALL' in log and 'EXIT=-13' in log and any(cmd in log for cmd in ['COMM="CAT"', 'COMM="HEAD"', 'COMM="CP"']):
        return ['ALTO']
        
    # 2. ESCALADA: Usuario no autorizado (UID > 1000) intentando usar sudo o su
    elif 'TYPE=SYSCALL' in log and auid_num > 1000 and any(cmd in log for cmd in ['COMM="SUDO"', 'COMM="SU"']):
        return ['ALTO', 'MEDIO']
        
    # 3. MANTENIMIENTO LEGÍTIMO: El sistema operativo autenticando usuarios
    elif 'TYPE=SYSCALL' in log and any(cmd in log for cmd in ['COMM="SSHD"', 'COMM="LOGIN"', 'COMM="(SYSTEMD)"', 'COMM="CRON"']):
        return ['INFO', 'BAJO', 'INFORMACION', 'INFORMADO']
        
    # 4. ADMIN LEGÍTIMO: Administrador original del sistema (AUID 1000) usando sudo
    elif 'TYPE=SYSCALL' in log and auid_num == 1000 and 'COMM="SUDO"' in log:
        return ['INFO', 'BAJO', 'INFORMACION', 'INFORMADO']
        
    # 5. Todo el ruido restante (CWD, PATH normales, PROCTITLE)
    else:
        return ['INFO', 'BAJO', 'INFORMACION', 'INFORMADO']

def evaluar_por_orden(path_completo, path_evaluar, funcion_heuristica):
    """
    Evalúa la precisión comparando fila por fila basándose en el orden estricto de los CSV.
    """
    df_completo = pd.read_csv(path_completo)
    df_evaluar = pd.read_csv(path_evaluar)
    
    # Verificación de integridad secuencial
    if len(df_completo) != len(df_evaluar):
        print(f"¡ALERTA!: Archivos desfasados. Completo: {len(df_completo)} | A evaluar: {len(df_evaluar)}")
        return
        
    # Construir la lista de la verdad absoluta
    verdad_en_orden = df_completo['Log'].apply(funcion_heuristica).tolist()
    
    aciertos = 0
    total_validos = 0
    
    # Comparar fila por fila
    for i in range(len(df_evaluar)):
        prediccion_llm = str(df_evaluar.loc[i, 'Riesgo']).upper().strip()
        etiquetas_reales = verdad_en_orden[i]
        
        if 'ERROR' not in etiquetas_reales:
            total_validos += 1
            if prediccion_llm in etiquetas_reales:
                aciertos += 1
                
    precision = (aciertos / total_validos) * 100 if total_validos > 0 else 0
    print(f"Precisión para {path_evaluar}: {precision:.2f}% ({aciertos}/{total_validos})")
    return precision



In [ ]:
# --- EJECUCIÓN ---
# Definir la ruta del archivo que tiene la verdad (RAW Completo)
ruta_verdad = '../results/prompt1/escenario5_resultados_raw_completo_phi3mini.csv'

# Evalur el RAW Completo contra sí mismo para la nota base
evaluar_por_orden(ruta_verdad, ruta_verdad, ground_truths_escenario5)

# Eevaluar los otros formatos
evaluar_por_orden(ruta_verdad, '../results/prompt1/escenario5_resultados_raw_reducido_phi3mini.csv', ground_truths_escenario5)
evaluar_por_orden(ruta_verdad, '../results/prompt1/escenario5_resultados_json_reducido_phi3mini.csv', ground_truths_escenario5)
evaluar_por_orden(ruta_verdad, '../results/prompt2/escenario5_resultados_raw_reducido_phi3mini.csv', ground_truths_escenario5)
evaluar_por_orden(ruta_verdad, '../results/prompt3/escenario5_resultados_raw_reducido_phi3mini.csv', ground_truths_escenario5)

Precisión para ../results/prompt1/escenario5_resultados_raw_completo_phi3mini.csv: 73.08% (171/234)
Precisión para ../results/prompt1/escenario5_resultados_raw_reducido_phi3mini.csv: 73.08% (171/234)
Precisión para ../results/prompt1/escenario5_resultados_json_reducido_phi3mini.csv: 62.39% (146/234)
Precisión para ../results/prompt2/escenario5_resultados_raw_reducido_phi3mini.csv: 57.69% (135/234)
Precisión para ../results/prompt3/escenario5_resultados_raw_reducido_phi3mini.csv: 67.09% (157/234)


67.09401709401709